In [0]:
%sql
CREATE OR REPLACE TEMP VIEW episode_candidate AS
SELECT
  co.person_id,
  MIN(co.condition_start_date) AS episode_start_date,
  CAST(MIN(co.condition_start_date) AS TIMESTAMP) AS episode_start_datetime,
  CAST(NULL AS DATE) AS episode_end_date,
  CAST(NULL AS TIMESTAMP) AS episode_end_datetime,
  CAST(NULL AS BIGINT) AS episode_parent_id,
  CAST(1 AS INT) AS episode_number,
  CAST(201826 AS INT) AS episode_object_concept_id, -- Type 2 diabetes mellitus
  CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'episode',
    't2dm_outpatient',
    CAST(co.person_id AS STRING),
    CAST(MIN(co.condition_start_date) AS STRING)
  ) AS episode_source_value
FROM _exponent.omop_tw.condition_occurrence co
WHERE co.condition_concept_id = 201826
GROUP BY co.person_id;

In [0]:
%sql
INSERT INTO _exponent.omop_tw.episode (
  person_id,
  episode_concept_id,
  episode_start_date,
  episode_start_datetime,
  episode_end_date,
  episode_end_datetime,
  episode_parent_id,
  episode_number,
  episode_object_concept_id,
  episode_type_concept_id,
  episode_source_value,
  episode_source_concept_id
)
SELECT
  ec.person_id,
  disease_episode.concept_id AS episode_concept_id,
  ec.episode_start_date,
  ec.episode_start_datetime,
  ec.episode_end_date,
  ec.episode_end_datetime,
  ec.episode_parent_id,
  ec.episode_number,
  ec.episode_object_concept_id,
  32817 AS episode_type_concept_id,   -- EHR record
  ec.episode_source_value,
  0 AS episode_source_concept_id
FROM episode_candidate ec
CROSS JOIN (
  SELECT concept_id
  FROM _exponent.omop.concept
  WHERE LOWER(concept_name) = 'disease episode'
    AND invalid_reason IS NULL
  LIMIT 1
) disease_episode;